In [1]:
# ============================================================
# BUOC 1: KET NOI VOI GOOGLE DRIVE
# Khi chay dong nay, Colab se hien len mot bang yeu cau ban
# cap quyen truy cap vao Drive. Hay bam dong y.
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import requests
import json
import os
import glob
import time
from datetime import datetime

# ============================================================
# BUOC 2: DINH NGHIA DUONG DAN TRUC TIEP TREN DRIVE
# ============================================================
# Tro thang vao thu muc ban da tao tren Google Drive
raw_data_dir = '/content/drive/MyDrive/steam_review_project/data/raw'
reviews_dir = os.path.join(raw_data_dir, 'reviews')

# Tao thu muc neu chua ton tai
os.makedirs(raw_data_dir, exist_ok=True)
os.makedirs(reviews_dir, exist_ok=True)

# ============================================================
    # BUOC 3 (DA CHINH SUA): DOC VA GOP TAT CA CAC FILE GAME
    # ============================================================
list_of_files = glob.glob(os.path.join(raw_data_dir, 'steam_games_raw_*.json'))
games_list = [] # Danh sach tong de chua tat ca game

if not list_of_files:
    print("Loi: Khong tim thay file danh sach game nao tren Drive.")
else:
    print(f"Tim thay {len(list_of_files)} file. Dang gop du lieu...")

        # Doc tung file va do tat ca vao danh sach tong
    for file_path in list_of_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            games_list.extend(data)

        # Loai bo cac game bi trung lap bang Dictionary (chia khoa la app_id)
    unique_games = {game['app_id']: game for game in games_list}
    games_list = list(unique_games.values())

    print(f"Da gop xong. Tong cong co {len(games_list)} tua game doc lap de cao binh luan.")

    # ============================================================
    # BUOC 4: CAO DU LIEU REVIEWS QUA STEAM API
    # ============================================================
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

    for game in games_list:
        app_id = game.get('app_id')
        title = game.get('title_raw')

        if not app_id:
            continue

        print(f"\n--- Dang xu ly game: {title} (App ID: {app_id}) ---")

        reviews_data = []
        cursor = '*'
        max_reviews_per_game = 500 # Colab co mang rat nhanh, ban co the tang so nay len cao hon

        while len(reviews_data) < max_reviews_per_game:
            url = f"https://store.steampowered.com/appreviews/{app_id}"

            params = {
                'json': 1,
                'filter': 'recent',
                'language': 'english',
                'review_type': 'all',
                'purchase_type': 'all',
                'num_per_page': 100,
                'cursor': cursor
            }

            try:
                response = requests.get(url, params=params, headers=headers, timeout=10)
                if response.status_code != 200:
                    print(f"Loi truy cap API. Ma loi: {response.status_code}")
                    break

                data = response.json()

                if 'reviews' not in data or len(data['reviews']) == 0:
                    print("Da het danh gia.")
                    break

                for item in data['reviews']:
                    review_dict = {
                        "review_id": item.get('recommendationid'),
                        "review_text_raw": item.get('review', ''),
                        "voted_up": item.get('voted_up'),
                        "playtime_forever": item.get('author', {}).get('playtime_forever', 0)
                    }
                    reviews_data.append(review_dict)

                cursor = data.get('cursor')
                print(f"Da thu thap duoc {len(reviews_data)} danh gia...")

                time.sleep(1)

            except Exception as e:
                print(f"Xay ra loi: {e}")
                break

        # ============================================================
        # BUOC 5: LUU FILE TRUC TIEP LEN GOOGLE DRIVE
        # ============================================================
        if reviews_data:
            file_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            review_file_name = f"{app_id}_reviews_raw_{file_timestamp}.json"
            review_file_path = os.path.join(reviews_dir, review_file_name)

            with open(review_file_path, 'w', encoding='utf-8') as f:
                json.dump(reviews_data, f, ensure_ascii=False, indent=4)

            print(f"Luu thanh cong {len(reviews_data)} danh gia vao Drive: {review_file_name}")

Mounted at /content/drive
Tim thay 1 file. Dang gop du lieu...
Da gop xong. Tong cong co 50 tua game doc lap de cao binh luan.

--- Dang xu ly game: Counter-Strike 2 (App ID: 730) ---
Da thu thap duoc 100 danh gia...
Da thu thap duoc 200 danh gia...
Da thu thap duoc 300 danh gia...
Da thu thap duoc 400 danh gia...
Da thu thap duoc 500 danh gia...
Luu thanh cong 500 danh gia vao Drive: 730_reviews_raw_20260605_100832.json

--- Dang xu ly game: Forza Horizon 6 (App ID: 2483190) ---
Da thu thap duoc 100 danh gia...
Da thu thap duoc 200 danh gia...
Da thu thap duoc 300 danh gia...
Da thu thap duoc 400 danh gia...
Da thu thap duoc 500 danh gia...
Luu thanh cong 500 danh gia vao Drive: 2483190_reviews_raw_20260605_100839.json

--- Dang xu ly game: 007 First Light (App ID: 3768760) ---
Da thu thap duoc 100 danh gia...
Da thu thap duoc 200 danh gia...
Da thu thap duoc 300 danh gia...
Da thu thap duoc 400 danh gia...
Da thu thap duoc 500 danh gia...
Luu thanh cong 500 danh gia vao Drive: 376876

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
